In [0]:
df = spark.read.csv("/Volumes/dbr_dev/pkustra555/datasets/social_media_screentime_mental_health_2026.csv", header=True, inferSchema=True)

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.pkustra555.social_media_screentime_mental_health")

Basic Spark DataFrame Operations

In [0]:
#won't be useful for future analysis 
df = df.drop("participant_id")

In [0]:
display(df.select("age","gender","daily_screen_hours"))

In [0]:
display(df.filter(df.daily_screen_hours>12))

In [0]:
from pyspark.sql.functions import col,avg, round, count, min, max
display(df.filter(col("gender").isNotNull()).groupBy("gender").agg(round(avg("daily_screen_hours"),2).alias("avg_screen_hours")))

In [0]:
display(df.groupBy("most_used_platform").agg(round(avg("anxiety_score_0to27"),2).alias("avg_stress")).orderBy("avg_stress", ascending=False))

In [0]:
platform_stats = (df.groupBy("most_used_platform").agg(round(avg("daily_screen_hours"),2).alias("avg_screen_time")).orderBy("avg_screen_time", ascending=False))

display(platform_stats)

In [0]:
display(df.groupBy("occupation").agg(round(avg("daily_screen_hours"), 2).alias("avg_scrren_time_per_occ")).orderBy("avg_scrren_time_per_occ"), ascending=False)

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.groupBy("most_used_platform").agg(min("daily_screen_hours").alias("min_hours"), max("daily_screen_hours").alias("max_hours")))

In [0]:
display(df.groupBy("most_used_platform").agg(count("*").alias("users")).orderBy("users", ascending=False))

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.filter(df.occupation=="Full-time employed").groupBy("most_used_platform").agg(round(avg("daily_screen_hours"),2).alias("avg_hours")))

Databricks visualization. Run in Databricks to view.

In [0]:
platform_users = df.groupBy("most_used_platform").agg(count('*').alias("user_count"))

In [0]:
joined_df = platform_stats.join(platform_users, on = "most_used_platform", how = "inner")
display(joined_df)

Dashboard was created by using the Databricks Dashboards tool. 

Loading data from an external API

In [0]:
import requests
from pyspark.sql import Row 

url = "http://universities.hipolabs.com/search"

params = { "country": "Poland"}

response = requests.get(url, params = params)
response.raise_for_status()
data = response.json()

print(type(data))
display(data)

In [0]:
import json 

rows = []

for university in data:
    rows.append(
        Row(
            name=university.get("name"),
            domains=university.get("domains"),
            web_pages= university.get("web_pages")
        )
    )

universities_df = spark.createDataFrame(rows)
display(universities_df)

DELTA LAKE BENEFITS(ACID, time travel, schema enforcement)

**ACID** 

A - atomicity - an operation is completed entirely or not at all (Example: bank transaction - money cannot disappear from one account without being added to the other account)

C - consistency - the data always remains correct and consistent

I - Isolation - many users can work on the same table at the same time without any conflicts (many people can shop in an online store at the same time)

D - durability - once the data is saved, it won't be lost, even if the system fails.

**time travel** - delta lake stores the history of table changes, every can creates a new version of the table, so it is possible to come back to the previous versions, compare different versions or recover deleted or modified data.

By version:
```sql
select * from history VERSION AS OF 2; 
```

By timestamp: 
```sql
select * from history TIMESTAMP AS OF '2026-07-01 12:00:00';
```

**schema enforcement** - delta lake checks that the schema of new data matches of the existing table. If the schemas are different, the attempt of operation is blocked. During every write operation, the schema of the new data is compared with the schema stored in the Data Log. This allow to maintain the good quality of the data.
**schema evolution** - if we want to add new columns, we can allow autoomatic schema updates.

**delta log** - every change is stored in the Delta log. It keeps track of the current table version, enables ACID transactions, allows Time Travel, and stores the table schema.